# Lecture 14 - Inheritance (https://bit.ly/intro_python_14)

* Where the default `__str__` and `__eq__` came from: every class inherits from `object`
* Inheritance: child classes, overriding methods, and `super()`
* Object composition
* Inheritance vs. composition, aka "is a" vs. "has a"
* **Study ahead:** work through on your own: An Example of OOP: Implementing the card game Old Maid

# Warm-up: a mystery from last lecture

Here is a class with nothing in it. Before running the cell, predict what each line will do.

In [ ]:
class Nothing:
  pass

n = Nothing()

print(n)         # we never wrote a __str__ method...
print(n == n)    # ...or an __eq__ method...
print(len(dir(n)), "attributes")  # ...so where did all of these come from?

Last lecture we *replaced* the default `__str__` and `__eq__` with our own versions. Today we'll see where the defaults live, and that replacing them was our first use of inheritance.

# Inheritance

* Last time we looked at polymorphism, and how the same syntax, e.g. `print(x)` or `x == y`, can do different things depending on the objects involved.

* We did this by redefining methods like `__str__` and `__eq__`. Every class we write gets default versions of those methods because every class *inherits* from a built-in class called `object`.

* Let's look at inheritance in general, starting with a silly example: pets.

In [2]:
class Pet:
  """ A class to represent a pet """

  def __init__(self, pet_name, age):
    self.pet_name, self.age = pet_name, age

  def noise(self):
    """ The noise the pet makes """
    return "??"

  def __str__(self):
    return f"Pet name: {self.pet_name}, age: {self.age}"

Suppose we wanted to make more specific objects to represent pet cats and dragons, but wanted to keep the basics of the Pet class the same.

One option would be to copy the code. As I've said before, this is a bad pattern: copying means we have twice the code, and when bugs crop up we have to fix them twice.

Instead we can use inheritance:

In [3]:
class Cat(Pet):  # The bracket notation after the class name says that Cat
                 # "inherits" from Pet
  def noise(self):
    return "meow"

class Dragon(Pet):
  def noise(self):
    return "rumble... growl..."

r = Pet("Binky", 40)
p = Cat("Tibbles", 5)
q = Dragon("Drogon", 2)

print(r, "goes", r.noise())
print(p, "goes", p.noise())
print(q, "goes", q.noise())

Pet name: Binky, age: 40 goes ??
Pet name: Tibbles, age: 5 goes meow
Pet name: Drogon, age: 2 goes rumble... growl...


Notice that Cat and Dragon never defined `__init__` or `__str__`, yet both work. They were inherited from Pet.

And note that we have a mixture of pets (Binky, Tibbles, etc.). We can treat them all as Pets, and each one still makes its own noise.

# So what is going on?

* When you inherit from a class, you create a *child* class (also called a subclass) of the *parent* class (also called the base class or superclass).

* A child class has all the functionality of the parent, but you are free to add or redefine methods and variables.

* When you call a method, Python starts looking in the object's own class. If the method isn't there, it looks in the parent, then the parent's parent, and so on.

* Redefining a method in a child class is called *overriding* it. This is what we were doing last lecture when we wrote `__str__`: the default version lives in `object`, and we overrode it.

Revisiting our picture from last time but now adding our new terminology using inheritance:

<img src="https://raw.githubusercontent.com/benedictpaten/intro_python/main/lecture_notebooks/figures/graffles/inheritance.jpg" width=600 height=300 />

* e.g. Calling `noise()` on a Cat finds the version in Cat and stops there. Calling `__str__()` on a Cat finds nothing in Cat, so Python moves up to Pet.

Looking under the hood a little bit, you can ask Python for the exact order it searches. It's called the *method resolution order*, or MRO:

In [5]:
print(Cat.__mro__)
print(Nothing.__mro__)  # even our empty class from the warm-up inherits from object

(<class '__main__.Cat'>, <class '__main__.Pet'>, <class 'object'>)
(<class '__main__.Nothing'>, <class 'object'>)


And now we can answer the warm-up: `object` is where the default `__str__`, `__eq__` and all the rest live.

In [6]:
print(object.__str__(p))  # the default __str__ that Pet overrode

print(p)                  # Pet's version, which Cat inherited

Pet name: Tibbles, age: 5


## Checking an object's type when there is inheritance

Last lecture we used `isinstance()` inside `__eq__`. With inheritance it does something sensible: a Cat *is a* Pet, so a Cat passes both checks.

In [7]:
p = Cat("Tibbles", 5)

print(isinstance(p, Cat))
print(isinstance(p, Pet))     # True: every Cat is a Pet
print(isinstance(p, Dragon))  # False
print(isinstance(p, object))  # True: everything is an object

True
True
False
True


**Puzzle:** what will `type(p) == Pet` print?

In [8]:
print(type(p))
print(type(p) == Pet)

<class '__main__.Cat'>
False


`type()` tells you the exact class, with no regard for inheritance. That's why `isinstance()` is almost always what you want: code that accepts a Pet should accept a Cat too.

# Challenge 1

In [ ]:
# Create a Hamster class that inherits from Pet and implements the noise method.
# Feel free to make up whatever noise you think a Hamster makes.


# This code should exercise the Hamster class
h = Hamster("moley", 1)
print(h, "goes", h.noise())

# Overriding `__init__()`

When defining a child class you often want to override `__init__` to add new variables. You don't want to copy the parent class's constructor; instead you call it, using `super()`:

In [10]:
class Snake(Pet):
  def __init__(self, pet_name, age, venomous=True):
    super().__init__(pet_name, age)  # super() finds the parent class, so this runs Pet.__init__

    # You could equivalently name the parent class explicitly, but then you have
    # to remember to pass self:
    # Pet.__init__(self, pet_name, age)

    self.venomous = venomous

  def __str__(self):
    return f"Snake name: {self.pet_name}, age: {self.age}, venomous?: {self.venomous}"

  def noise(self):
    return "hiss"

s = Snake("Cuddly", 1)

print(s, "goes", s.noise())

Snake name: Cuddly, age: 1, venomous?: True goes hiss


**Take home:** use `super().__init__(...)` (don't forget the parentheses after `super`) to call the parent constructor, and do it first, before doing anything else in the child's `__init__()`.

## What goes wrong if you forget

This is one of the most common bugs when people start using inheritance. Predict the error before running:

In [ ]:
class Parrot(Pet):
  def __init__(self, pet_name, age, vocabulary):
    self.vocabulary = vocabulary   # forgot to call super().__init__(pet_name, age)

polly = Parrot("Polly", 3, ["hello", "pieces of eight"])
print(polly)

Pet's `__str__` expects `self.pet_name` to exist, but only Pet's `__init__` creates it, and we never called it.

## Extending a method instead of replacing it

`super()` isn't only for `__init__`. Often you want the parent's behaviour plus a little extra. Call the parent's version, then build on its result:

In [12]:
class Dragon(Pet):
  def __init__(self, pet_name, age, wingspan):
    super().__init__(pet_name, age)
    self.wingspan = wingspan

  def noise(self):
    return "rumble... growl..."

  def __str__(self):
    # Reuse Pet's __str__ and add to it, rather than retyping it
    return super().__str__() + f", wingspan: {self.wingspan}m"

d = Dragon("Drogon", 2, 12)
print(d)

Pet name: Drogon, age: 2, wingspan: 12m


# Challenge 2

In [ ]:
# Expand your Hamster class from Challenge 1 to include a constructor (__init__) method
# which calls the parent
# constructor. The constructor should take the pet_name and age attributes and a
# favorite_food argument, which will stored as an object variable of the same name.


# This code should work
h = Hamster("moley", 1, "oats")
assert h.pet_name == "moley"
assert h.age == 1
assert h.favorite_food == "oats"

# Inheritance in the wild

Python itself is built this way. Two examples you have already been using.

**`bool` is a child class of `int`.** `True` and `False` are really the integers 1 and 0 with a nicer `__str__`:

In [14]:
print(bool.__mro__)
print(isinstance(True, int))
print(True + True)              # a party trick, but it explains why this works...
print(sum([True, False, True]))  # ...and why this is a handy way to count things

(<class 'bool'>, <class 'int'>, <class 'object'>)
True
2
2


**Errors form a family tree.** Every error you've seen is a class, and they inherit from each other:

In [15]:
print(ZeroDivisionError.__mro__)
print(issubclass(ZeroDivisionError, ArithmeticError))  # issubclass is isinstance for classes
print(issubclass(KeyError, LookupError), issubclass(IndexError, LookupError))

(<class 'ZeroDivisionError'>, <class 'ArithmeticError'>, <class 'Exception'>, <class 'BaseException'>, <class 'object'>)
True
True True


Soon we will visit exceptions, and this tree is what lets you say "catch any arithmetic problem" without listing every kind.

# Object composition

When you start writing OOP code, you may be tempted to create really large objects.

However, just like with functions, in code small is generally beautiful. Put another way, it is often better to build more, simple objects and compose them together than to build fewer, more complex objects.

Consider this very simple example from the textbook that we looked at last time:

In [16]:
# Recall from last time that we created a simple class for
# representing 2D coordinates:

class Point:
  """ Create a new Point, at coordinates x, y """

  def __init__(self, x=0, y=0):
    """ Create a new point at x, y """
    self.x = x
    self.y = y

  def distance_from_origin(self):
    """ Compute my distance from the origin """
    return ((self.x ** 2) + (self.y ** 2)) ** 0.5  # Pythagoras' theorem

  def __str__(self):
    """Return a string representing the point"""
    return f"({self.x}, {self.y})"

Okay, suppose we wanted to make a line object, defined by a pair of points, one for each end:

In [17]:
class Line:
  """ A class to represent lines """

  def __init__(self, p1, p2):
    """ Initialize line with two Point objects, p1 and p2, which
    represent the two ends of the line
    """
    self.p1 = p1
    self.p2 = p2

  def __str__(self):
    return f"({self.p1}, {self.p2})"  # Note how we reuse the __str__ method of Point

l1 = Line(Point(0, 0), Point(100, 200))
print("line: ", l1)

line:  ((0, 0), (100, 200))


This is an example of composition, because a line "has a" pair of points.

It is natural to do this with composition, in which we reuse Points to represent the two coordinate pairs.

# Challenge 3

In [ ]:
# Create a triangle class, defined by three points, p1, p2, and p3.
# Define the str method to print out the coordinates of each vertex (point) in the triangle.


# This code should work
t = Triangle(Point(0, 0), Point(100, 200), Point(300, 500))
print(t)  ## Should print out the three vertices (points) of the triangle

# Inheritance vs. composition, aka "is a" vs. "has a"

When you first learn about inheritance there is a tendency to overuse it. Consider: we could have implemented the line class using inheritance:

In [19]:
class Line(Point):
  """ A class to represent lines """

  def __init__(self, x, y, x2, y2):
    """ Initialize line with two sets of x, y coordinates: (x, y) and (x2, y2)
    """
    super().__init__(x, y)
    self.x2 = x2
    self.y2 = y2

  def __str__(self):
    return f"(({self.x}, {self.y}), ({self.x2}, {self.y2}))"

l1 = Line(0, 0, 100, 200)
print("line: ", l1)

line:  ((0, 0), (100, 200))


It runs, but a Line is not a kind of Point. It has ends which are points.

Practically, instead of composing a line out of a pair of points, we have overridden the behaviour of Point and kept almost nothing from it apart from the variables x and y. Worse, a Line now claims to be a Point:

In [20]:
print(isinstance(l1, Point))        # a line says it is a point...
print(l1.distance_from_origin())    # ...so this is allowed, but what does it mean for a line?

True
0.0


We also can't share points between lines without duplicating memory. This might seem trivial, but it can have major design implications.

**The take home:** when building object hierarchies, think carefully about whether the relationship between objects is an "is a" relationship or a "has a" relationship.

* "Is a" implies inheritance.
* "Has a" implies composition.

A line "has a" pair of points. A line "is not a" point.

# An Example of OOP: Implementing the card game Old Maid

You may want to read the following ahead of going through the remainder of this lecture to get the most out of the code walk-through.

We're going to use the example of a card game (Old Maid; adapted from http://openbookproject.net/thinkcs/python/english3e/inheritance.html, which contains a good write-up).

The objective is to study how a card game can be mapped to a set of objects in a natural way.

A card game has:

* Individual cards, each with a suit and rank, e.g. Ace, 2, 3, 4, ...
* One or more decks (a deck being a full set of cards)
* Hands (a hand is a subset of cards)
* Game logic

We will see that we can map each of these concepts to a class definition. Along the way we'll use both composition and inheritance, and at the end we'll use the finished program to answer a question: is Old Maid fair?

Let's first build a card class:

## The Card class

* A card has a suit (Clubs, Diamonds, Hearts or Spades) and a rank (Ace, 2, 3 ...).
* Cards also have a color: Diamonds and Hearts are red, Clubs and Spades are black. Since the suit implies the color, we don't need to store it.
* We implement the comparison methods from last lecture so that we can compare and sort cards.

In [21]:
class Card:
  """ Represents a card from a deck of cards. """

  # These are class variables: shared by all cards, so defined once here
  # rather than in __init__
  suits = ["Clubs", "Diamonds", "Spades", "Hearts"]
  ranks = ["narf", "Ace", "2", "3", "4", "5", "6", "7",
           "8", "9", "10", "Jack", "Queen", "King"]  # "narf" fills index 0 so that rank 1 is the Ace

  def __init__(self, suit=0, rank=0):
    """ Create a card using integers to represent suit and rank. """
    # A couple of handy asserts to check any cards we build make sense
    assert 0 <= suit < 4
    assert 0 <= rank < 14

    self.suit = suit
    self.rank = rank

  def __str__(self):
    # Looking up the suits/ranks lists gives a human readable card
    return self.ranks[self.rank] + " of " + self.suits[self.suit]

  def __repr__(self):
    return str(self)  # so that lists of cards print nicely too

  def same_color(self, other):
    """ True if the cards have the same color.
    Diamonds and Hearts are both red, Clubs and Spades are both black.
    """
    return self.suit == other.suit or self.suit == (other.suit + 2) % 4

  # Comparison, as in the last lecture: cards are compared first by suit, then by rank

  def sort_key(self):
    return (self.suit, self.rank)

  def __eq__(self, other):
    if not isinstance(other, Card):
      return NotImplemented
    return self.sort_key() == other.sort_key()

  def __lt__(self, other):
    return self.sort_key() < other.sort_key()

  def __le__(self, other):
    return self.sort_key() <= other.sort_key()

  def __gt__(self, other):
    return self.sort_key() > other.sort_key()

  def __ge__(self, other):
    return self.sort_key() >= other.sort_key()

In [22]:
# Play with the cards

x = Card(2, 1)
y = Card(2, 13)

print("x is:", x)
print("y is:", y)
print("x is less than y:", x < y)  # Aces low
print("same color?", x.same_color(Card(0, 5)))  # Spades and Clubs are both black
print(sorted([Card(3, 12), Card(0, 2), Card(2, 1)]))

x is: Ace of Spades
y is: King of Spades
x is less than y: True
same color? True
[2 of Clubs, Ace of Spades, Queen of Hearts]


## The Deck class

A deck is a set of cards.

We should be able to shuffle a deck. We also implement methods to remove cards from the deck so that we can deal them into hands.

A Deck is not a type of Card. A deck is composed of cards, so we use composition: the deck "has a" list of cards.

In [23]:
import random

class Deck:
  """ Represents a deck of cards. """

  def __init__(self):
    """ Creates a full set of cards """
    self.cards = []  # cards stores the list of cards
    for suit in range(4):  # Note the nested for loops
      for rank in range(1, 14):
        self.cards.append(Card(suit, rank))

  def __str__(self):
    s = ""  # Builds a picture of the deck, laying the cards out diagonally
    for i in range(len(self.cards)):
      s = s + " " * i + str(self.cards[i]) + "\n"
    return s

  def __len__(self):
    return len(self.cards)

  def shuffle(self):
    """ Shuffles the cards into a random order """
    # random.shuffle(self.cards) would do this in one line. Here is what it does inside:
    num_cards = len(self.cards)
    for i in range(num_cards):
      j = random.randrange(i, num_cards)  # pick a card from the not-yet-shuffled part...
      self.cards[i], self.cards[j] = self.cards[j], self.cards[i]  # ...and swap it into place

  def remove(self, card):
    """ Removes the card from the deck. Returns True if successful. """
    if card in self.cards:  # 'in' uses Card.__eq__ to find a matching card
      self.cards.remove(card)
      return True
    else:
      return False

  def pop(self):
    """ Removes the last card in the deck and returns it. """
    return self.cards.pop()

  def add(self, card):
    """ Adds the card to the deck. """
    assert card not in self.cards  # Protect against having the same card in the deck twice
    self.cards.append(card)

  def is_empty(self):
    """ Returns True if the deck is empty. """
    return self.cards == []

  def deal(self, hands, num_cards_per_hand=999):
    """ Deals cards into hands, one card at a time to each hand in turn,
    until each hand has num_cards_per_hand cards or the deck runs out.
    """
    num_hands = len(hands)
    for i in range(num_cards_per_hand * num_hands):
      if self.is_empty():
        break  # Out of cards
      card = self.pop()           # Take the top card
      hand = hands[i % num_hands]  # Whose turn is next?
      hand.add(card)              # Add the card to the hand

In [24]:
# Play with a deck of cards

deck = Deck()

print(deck)

print("There are", len(deck), "cards")

deck.shuffle()

print("After shuffling, the top five cards are:", deck.cards[-5:])

Ace of Clubs
 2 of Clubs
  3 of Clubs
   4 of Clubs
    5 of Clubs
     6 of Clubs
      7 of Clubs
       8 of Clubs
        9 of Clubs
         10 of Clubs
          Jack of Clubs
           Queen of Clubs
            King of Clubs
             Ace of Diamonds
              2 of Diamonds
               3 of Diamonds
                4 of Diamonds
                 5 of Diamonds
                  6 of Diamonds
                   7 of Diamonds
                    8 of Diamonds
                     9 of Diamonds
                      10 of Diamonds
                       Jack of Diamonds
                        Queen of Diamonds
                         King of Diamonds
                          Ace of Spades
                           2 of Spades
                            3 of Spades
                             4 of Spades
                              5 of Spades
                               6 of Spades
                                7 of Spades
                                 8 

## The Hand class

A hand of cards is a smaller collection of cards. We'll make it a child class of Deck, so it gets `pop`, `add`, `shuffle`, `is_empty` and `__len__` for free, and we add:

* a name (for the player)
* an adapted `__str__` method

In [25]:
class Hand(Deck):
  """ Represents a hand of cards. """

  def __init__(self, name=""):
    """ Creates an initially empty hand. """
    self.cards = []  # Note we deliberately don't call Deck.__init__, because that
                     # would fill the hand with all 52 cards
    self.name = name

  def __str__(self):
    s = "Hand " + self.name
    if self.is_empty():
      s += " is empty\n"
    else:
      s += " contains\n"
    return s + Deck.__str__(self)  # Reuse Deck's diagonal picture of the cards

In [26]:
# Play with hands of cards

deck = Deck()

deck.shuffle()  # Shuffle the deck

# Make two hands, dealing cards from the deck
daves = Hand("Dave")
rojins = Hand("Rojin")
deck.deal([daves, rojins], 5)

# Print the resulting hands
print(daves)
print(rojins)
print(len(deck), "cards left in the deck")

Hand Dave contains
4 of Spades
 7 of Hearts
  King of Diamonds
   9 of Diamonds
    10 of Diamonds

Hand Rojin contains
4 of Hearts
 8 of Spades
  Ace of Spades
   Queen of Spades
    6 of Spades

42 cards left in the deck


**Discussion:** is a Hand really a kind of Deck? Hand inherits `deal()`, so one hand can deal cards into another, which is a little odd. On the other hand, it also inherits `pop()`, `add()`, `shuffle()` and `is_empty()`, which turn out to be exactly what we need when players pass cards to each other in Old Maid. Design decisions like this are rarely clear-cut; the textbook chose inheritance, and we'll stick with it.

## The CardGame class

We need a class to represent the logic of the game.

This logic is not inherently part of the Card, Deck or Hand classes. It makes sense for it to be a separate class that encapsulates the rules.

Here we use the trick of creating a "base class", a mostly empty class that represents the basic functionality that any card game might need, but which in and of itself is not a complete game.

In [27]:
class CardGame:
  """ Represents a card game. Designed
  to be inherited by more specific, complete games
  """

  def __init__(self):
    self.deck = Deck()
    self.deck.shuffle()

## Old Maid

The mechanics of Old Maid (well, the version we're doing):

* 2 or more players
* A pair of cards is a match if they have equal color and rank
* The goal of play is to discard pairs of matched cards, and not to be left with the odd card (a queen) at the end.
* Setup:
  * One queen is removed from the deck
  * Remaining cards are dealt and players discard any pairs of matches in their hands
* Play:
  * The first player offers their hand face-down to the neighboring player on their left
  * That neighbor chooses one card and adds it to their hand, discarding any new pair of matches
  * The neighbor then offers their hand to the next player, and so forth
  * A player who runs out of cards is out of the game
  * Play continues until only one card remains, the unmatchable queen (called the old maid).
* The player left holding the old maid is the loser (weirdly, there is no overall winner).

## Old Maid Hand

To implement the removal of pairs of matches from a hand it makes sense to create a specialized version of a hand:

In [28]:
class OldMaidHand(Hand):
  """ Old Maid version of a hand. """

  def remove_matches(self, verbose=True):
    """ Remove any pairs of matches from the hand.
    Returns the removed cards as a list.
    """
    removed_cards = []
    for card in self.cards[:]:  # Note that we loop over a copy of self.cards,
                                # because we remove from self.cards inside the loop, and
                                # you can't edit and iterate over the same list at once
      match = Card((card.suit + 2) % 4, card.rank)  # The other card of the same color and rank
      if match in self.cards:
        self.cards.remove(card)
        self.cards.remove(match)
        removed_cards.append(card)
        removed_cards.append(match)
        if verbose:
          print(f"Hand {self.name}: {card} matches {match}")

    return removed_cards

In [29]:
# Make a shuffled deck
deck = Deck()
deck.shuffle()

# Make a hand, dealing cards from the deck
daves = OldMaidHand("Dave")
deck.deal([daves], 25)

# Print the resulting hand
print(daves)

# Remove the matches
removed_cards = daves.remove_matches()
print("Removed cards", removed_cards)

# Print the resulting hand
print(daves)

Hand Dave contains
8 of Diamonds
 Jack of Spades
  2 of Hearts
   Jack of Diamonds
    7 of Spades
     2 of Clubs
      10 of Spades
       Ace of Clubs
        4 of Hearts
         5 of Hearts
          4 of Spades
           Queen of Spades
            King of Hearts
             9 of Spades
              7 of Clubs
               6 of Clubs
                4 of Clubs
                 Ace of Hearts
                  2 of Spades
                   5 of Clubs
                    6 of Diamonds
                     3 of Diamonds
                      9 of Hearts
                       4 of Diamonds
                        2 of Diamonds

Hand Dave: 2 of Hearts matches 2 of Diamonds
Hand Dave: 7 of Spades matches 7 of Clubs
Hand Dave: 2 of Clubs matches 2 of Spades
Hand Dave: 4 of Hearts matches 4 of Diamonds
Hand Dave: 4 of Spades matches 4 of Clubs
Removed cards [2 of Hearts, 2 of Diamonds, 7 of Spades, 7 of Clubs, 2 of Clubs, 2 of Spades, 4 of Hearts, 4 of Diamonds, 4 of Spades, 4 of C

## Old Maid Game class

Okay, now let's implement the actual game logic:

In [30]:
class OldMaidGame(CardGame):
  def __init__(self, names, verbose=True):
    """ Construct a game of Old Maid. Each name in names is the name of a player.
    Set verbose=False to play silently.
    """
    super().__init__()  # Call the base class constructor, which makes and shuffles the deck
    self.verbose = verbose

    # Remove the Queen of Clubs (so the Queen of Spades is the Old Maid)
    self.deck.remove(Card(0, 12))

    # Make a hand for each player
    self.hands = []
    for name in names:
      self.hands.append(OldMaidHand(name))

  def say(self, *args):
    """ Print, but only if the game is verbose """
    if self.verbose:
      print(*args)

  def play(self):
    """ Play the game. Returns the name of the loser. """
    # Deal the cards
    self.deck.deal(self.hands)
    self.say("---------- Cards have been dealt")
    self.print_hands()

    # Remove initial matches
    count = 1  # Count of discarded cards (1 to start, representing the discarded queen)
    for hand in self.hands:
      count += len(hand.remove_matches(self.verbose))
    self.say("---------- Matches discarded, play ready to begin")
    self.print_hands()

    # Play until all 50 matchable cards are gone
    turn = 0
    num_hands = len(self.hands)
    while count < 51:
      count += self.play_one_turn(turn)
      turn = (turn + 1) % num_hands

    self.say("---------- Game is Over")
    self.print_hands()

    # Whoever still holds a card has the old maid
    for hand in self.hands:
      if not hand.is_empty():
        return hand.name

  def play_one_turn(self, i):
    """ Play one turn for the ith player. Returns the number of cards discarded. """
    hand = self.hands[i]

    if hand.is_empty():
      return 0  # A player with no cards is out of the game and skips their turn

    # Find a neighboring hand with one or more cards
    neighbor = self.hands[self.find_neighbor(i)]

    # Pick a random card from the neighbor (the neighbor's hand is shuffled, so
    # taking the top card is a random choice)
    picked_card = neighbor.pop()

    hand.add(picked_card)
    self.say("Hand", hand.name, "picked", picked_card, "from", neighbor.name)
    count = len(hand.remove_matches(self.verbose))

    hand.shuffle()  # Shuffle so the next player picks a random card from us

    return count

  def find_neighbor(self, i):
    """ Find the next player after player i who still has cards """
    num_hands = len(self.hands)
    for step in range(1, num_hands):
      neighbor = (i + step) % num_hands
      if not self.hands[neighbor].is_empty():
        return neighbor

  def print_hands(self):
    for hand in self.hands:
      self.say(hand)

In [31]:
game = OldMaidGame(["Allen", "Jeff", "Chris"])
loser = game.play()
print("The old maid ended up with", loser)

---------- Cards have been dealt
Hand Allen contains
6 of Diamonds
 3 of Spades
  7 of Diamonds
   Ace of Clubs
    Jack of Clubs
     9 of Clubs
      3 of Hearts
       10 of Hearts
        9 of Spades
         Ace of Spades
          4 of Spades
           4 of Diamonds
            8 of Hearts
             5 of Spades
              9 of Hearts
               4 of Hearts
                7 of Spades

Hand Jeff contains
5 of Clubs
 3 of Clubs
  3 of Diamonds
   Ace of Diamonds
    Jack of Diamonds
     10 of Spades
      Jack of Hearts
       9 of Diamonds
        10 of Clubs
         10 of Diamonds
          6 of Hearts
           King of Diamonds
            7 of Hearts
             2 of Diamonds
              8 of Clubs
               2 of Spades
                7 of Clubs

Hand Chris contains
King of Spades
 5 of Diamonds
  2 of Hearts
   Queen of Diamonds
    Ace of Hearts
     Queen of Hearts
      2 of Clubs
       4 of Clubs
        6 of Spades
         King of Clubs
          

## Is Old Maid fair?

Now we have a program that plays a whole game in a fraction of a second, we can ask questions that would take all day with real cards.

Does the player who picks first have an advantage, or a disadvantage? Or does it make no difference? Make a prediction, then run the simulation.

In [32]:
from collections import Counter

losses = Counter()
num_games = 3000

for i in range(num_games):
  game = OldMaidGame(["Allen", "Jeff", "Chris"], verbose=False)
  losses[game.play()] += 1

for name, count in sorted(losses.items()):
  print(f"{name:6} lost {count:5} games ({100 * count / num_games:.1f}%)")

Allen  lost  1282 games (42.7%)
Chris  lost  1029 games (34.3%)
Jeff   lost   689 games (23.0%)


Run it a couple more times to see how much the numbers move from run to run. Then some experiments:

* Try 2 players, then 4 or 6. Does the pattern hold?
* Can you work out *why* one seat does better? Think about who gains a card and who loses a card on the very first turn, and about the rule that a player with no cards is out. A good way to investigate is to make the game verbose for a single short game and read the log.

This is a general lesson: once the rules of something are written as code, you can test claims about it by experiment rather than argument.

# Cheat sheet

| You write... | ...and get |
|---|---|
| `class Child(Parent):` | a class with everything Parent has |
| `def method(self): ...` in Child | a new method, or an *override* if Parent has one with the same name |
| `super().__init__(...)` | the parent's constructor, run on this object (call it first) |
| `super().method()` | the parent's version of a method, so you can extend it rather than retype it |
| `isinstance(obj, Parent)` | True for a Parent *or any child class* object |
| `issubclass(Child, Parent)` | True if Child inherits from Parent |
| `Child.__mro__` | the order Python searches for a method |
| `self.part = Part(...)` | composition: this object *has a* Part |

Everything inherits from `object`, which is where the default `__init__`, `__str__`, `__repr__` and `__eq__` live.

# Reading

* Read Chapter 22 (inheritance): http://openbookproject.net/thinkcs/python/english3e/inheritance.html

# Homework

* Go to Canvas and complete the lecture quiz, which involves completing each challenge problem
* ZyBooks Reading 14

# Practice Problems

In [ ]:
"""
Problem 1: Basic Inheritance

Create a base class Book with:
- An __init__ method taking title, author and year parameters
- A get_info method that returns "Unknown format"
- A __str__ method that returns "{title} by {author} ({year})"

Then create an Ebook class that inherits from Book and:
- Overrides the get_info method to return "Digital format - Kindle compatible"
- Adds a new method get_download_link that returns "Download {title} at books.example.com/{title}"
"""

class Book:
  pass  # Add your code here

class Ebook(Book):
  pass  # Add your code here

# Tests
book = Ebook("Python Basics", "Jane Smith", 2023)
assert str(book) == "Python Basics by Jane Smith (2023)"
assert book.get_info() == "Digital format - Kindle compatible"
assert book.get_download_link() == "Download Python Basics at books.example.com/Python Basics"

In [ ]:
"""
Problem 2: Constructor Inheritance

Create a base class Appliance with:
- An __init__ method taking brand, model, and power_rating parameters
- A method energy_usage() that returns the power_rating

Create a class SmartAppliance that inherits from Appliance and:
- Adds a wifi_enabled parameter to __init__
- Overrides energy_usage() to add 10% when wifi is enabled
"""

class Appliance:
  pass  # Add your code here

class SmartAppliance(Appliance):
  pass  # Add your code here

# Tests
smart_fridge = SmartAppliance("Samsung", "RF28", 150, True)
assert smart_fridge.energy_usage() == 165  # 150 + 10%
regular_fridge = SmartAppliance("LG", "LF22", 150, False)
assert regular_fridge.energy_usage() == 150  # No wifi, no extra power

In [ ]:
"""
Problem 3: Biological Sequences

The parent class does the work; the child classes supply the data. Complete the classes so that:

- Sequence has __len__, __str__ (returns the bases) and a method complement() that returns a NEW
  object of the SAME class as self, with each base replaced using self.complement_map.
  Hint: type(self)(new_bases) builds a new object of whatever class self is.
- Sequence also has reverse_complement(), which returns the complement, reversed.
- DNASequence inherits from Sequence and defines the class variable
  complement_map = {"A": "T", "T": "A", "G": "C", "C": "G"}
  and a method transcribe() that returns an RNASequence with every T replaced by U.
- RNASequence inherits from Sequence and defines complement_map = {"A": "U", "U": "A", "G": "C", "C": "G"}
"""

class Sequence:
  def __init__(self, bases):
    self.bases = bases

  # Add your methods here

class DNASequence(Sequence):
  pass  # Add your code here

class RNASequence(Sequence):
  pass  # Add your code here

# Tests
dna = DNASequence("GATTACA")
assert len(dna) == 7
assert str(dna) == "GATTACA"
assert str(dna.complement()) == "CTAATGT"
assert str(dna.reverse_complement()) == "TGTAATC"
assert isinstance(dna.complement(), DNASequence)

rna = dna.transcribe()
assert isinstance(rna, RNASequence)
assert str(rna) == "GAUUACA"
assert str(rna.reverse_complement()) == "UGUAAUC"
assert isinstance(rna.reverse_complement(), RNASequence)

In [ ]:
"""
Problem 4: Poker Hand Evaluator Problem (THIS IS A BIG ONE)

Your task is to implement a (simplified) PokerHand class that can evaluate standard 5-card poker hands.
The PokerHand class should inherit from the Hand class and implement methods to:
1. Detect different types of poker hands (pairs, straights, etc.)
2. Compare hands to determine winners
3. Score hands according to poker rules

Remember:
- A poker hand consists of exactly 5 cards
- Card ranks are: 2-10, Jack(11), Queen(12), King(13), Ace(1 or 14)
- Card suits are: Hearts(0), Diamonds(1), Clubs(2), Spades(3)
- Hand rankings from lowest to highest are:
  * High Card (1)
  * One Pair (2)
  * Two Pair (3)
  * Three of a Kind (4)
  * Straight (5)
  * Flush (6)
  * Full House (7)
  * Four of a Kind (8)
  * Straight Flush (9)
  * Royal Flush (10)
"""

class Card:
  """A playing card, simplified from the lecture examples"""
  suits = ["Hearts", "Diamonds", "Clubs", "Spades"]
  ranks = ["narf", "Ace", "2", "3", "4", "5", "6", "7",
           "8", "9", "10", "Jack", "Queen", "King"]

  def __init__(self, suit=0, rank=0):
    self.suit = suit
    self.rank = rank

  def __str__(self):
    return f"{self.ranks[self.rank]} of {self.suits[self.suit]}"

  def __repr__(self):
    return str(self)

class Hand:
  """A hand of cards, simplified from the lecture examples"""
  def __init__(self):
    self.cards = []

  def add_card(self, card):
    self.cards.append(card)

  def __str__(self):
    return ", ".join(str(card) for card in self.cards)

class PokerHand(Hand):
  """
  A poker hand class that can evaluate poker hands

  Required methods to implement:

  1. Hand Detection Methods (all return bool):
    - has_pair(self): Returns True if hand contains one or more pairs
    - has_two_pair(self): Returns True if hand contains exactly two pairs
    - has_three_kind(self): Returns True if hand contains three of a kind
    - has_straight(self): Returns True if hand contains 5 consecutive ranks
    - has_flush(self): Returns True if all cards are the same suit
    - has_full_house(self): Returns True if hand has three of kind and pair
    - has_four_kind(self): Returns True if hand contains four of a kind
    - has_straight_flush(self): Returns True if hand is straight and flush
    - has_royal_flush(self): Returns True if hand is 10-A straight flush

  2. Scoring Methods:
    - get_score(self) -> int: Returns 1-10 based on hand rank (1=high card, 10=royal flush)
      * High Card (1)
      * One Pair (2)
      * Two Pair (3)
      * Three of a Kind (4)
      * Straight (5)
      * Flush (6)
      * Full House (7)
      * Four of a Kind (8)
      * Straight Flush (9)
      * Royal Flush (10)
    - get_high_card(self) -> Card: Returns highest card in hand

  3. Comparison Methods:
    - __eq__(self, other): True if hands are equal
      Two hands will be considered equal if they have the same score via get_score() and the same highest card.
    - __lt__(self, other): True if this hand scores lower than the other or has equal score and a
      lower valued highest card, judged by rank and (if tied) then suit.
    - __gt__(self, other): True if this hand scores higher than the other or has equal score and a
      higher valued highest card, judged by rank and (if tied) then suit.

  Hints: it is worth thinking about helper functions to avoid copying and pasting code
  across your implemented methods!
  """
  def __init__(self):
    super().__init__()



# Test cases
def test_poker_hands_extended():
  # Test straight
  straight = PokerHand()
  for i, rank in enumerate([7, 8, 9, 10, 11]):  # 7 through Jack
    straight.add_card(Card(i % 2, rank))
  assert straight.has_straight()
  assert straight.get_score() == 5

  # Test Ace-low straight
  ace_low = PokerHand()
  ace_low.add_card(Card(0, 1))  # Ace
  for i, rank in enumerate([2, 3, 4, 5]):
    ace_low.add_card(Card(i % 2, rank))
  assert ace_low.has_straight()

  # Test flush
  flush = PokerHand()
  for rank in [2, 5, 7, 9, 11]:  # Random ranks, same suit
    flush.add_card(Card(0, rank))
  assert flush.has_flush()
  assert flush.get_score() == 6

  # Test full house
  full = PokerHand()
  for suit in range(3):  # Three 8s
    full.add_card(Card(suit, 8))
  for suit in range(2):  # Two 4s
    full.add_card(Card(suit, 4))
  assert full.has_full_house()
  assert full.get_score() == 7

  # Test hand comparisons with tie breaks
  high_pair = PokerHand()
  high_pair.add_card(Card(0, 13))  # Pair of Kings
  high_pair.add_card(Card(1, 13))
  high_pair.add_card(Card(0, 5))
  high_pair.add_card(Card(1, 4))
  high_pair.add_card(Card(2, 3))

  low_pair = PokerHand()
  low_pair.add_card(Card(0, 12))  # Pair of Queens
  low_pair.add_card(Card(1, 12))
  low_pair.add_card(Card(0, 5))
  low_pair.add_card(Card(1, 4))
  low_pair.add_card(Card(2, 3))

  assert high_pair > low_pair
  assert low_pair < high_pair
  assert not high_pair == low_pair

  print("Extended tests passed!")

test_poker_hands_extended()